In [1]:
import tkinter as tk
from tkinter import messagebox
import pymysql
import re
import webbrowser
import os
from datetime import datetime

class DictionaryApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Python 수업 용어 사전")
        self.root.geometry("500x900") # 버튼 추가로 인한 높이 조절
        
        # 설정 및 폰트
        self.config_file = "db_config.txt"
        self.font_main = ("Malgun Gothic", 10)
        self.font_bold = ("Malgun Gothic", 11, "bold")
        self.font_title = ("Malgun Gothic", 15, "bold")
        self.font_info = ("Malgun Gothic", 9, "italic")
        self.font_ticker = ("Malgun Gothic", 9)
        
        self.db_host = "localhost"
        self.connection = None
        self.current_user = ""
        self.ticker_timer = None 
        
        # 검색 히스토리 및 전광판 관련
        self.history = []
        self.history_index = -1
        self.recent_terms = []
        self.current_ticker_idx = 0
        
        self.check_auto_login()

    def check_auto_login(self):
        if os.path.exists(self.config_file):
            try:
                with open(self.config_file, "r", encoding="utf-8") as f:
                    lines = f.readlines()
                    saved_id = lines[0].strip()
                    saved_pw = lines[1].strip()
                if self.perform_login(saved_id, saved_pw, silent=True):
                    return
            except: pass
        self.show_login_ui()

    def show_login_ui(self):
        if hasattr(self, 'search_frame'): self.search_frame.destroy()
        if hasattr(self, 'ticker_frame'): self.ticker_frame.destroy()
        
        self.login_frame = tk.Frame(self.root)
        self.login_frame.pack(pady=50)
        tk.Label(self.login_frame, text="🐍 Python 수업 용어 사전", font=self.font_title).pack(pady=20)
        
        tk.Label(self.login_frame, text="Linux ID:", font=self.font_main).pack()
        self.ent_user = tk.Entry(self.login_frame, font=self.font_main)
        self.ent_user.pack(pady=5)
        self.ent_user.bind('<Return>', lambda e: self.ent_pw.focus())
        
        tk.Label(self.login_frame, text="Password:", font=self.font_main).pack()
        self.ent_pw = tk.Entry(self.login_frame, show="*", font=self.font_main)
        self.ent_pw.pack(pady=5)
        self.ent_pw.bind('<Return>', lambda e: self.btn_login_click())
        
        tk.Button(self.login_frame, text="접속", font=self.font_bold, command=self.btn_login_click, 
                  bg="#2196F3", fg="white", width=20).pack(pady=20)

    def btn_login_click(self):
        user_id = self.ent_user.get().strip()
        user_pw = self.ent_pw.get().strip()
        if self.perform_login(user_id, user_pw):
            with open(self.config_file, "w", encoding="utf-8") as f:
                f.write(f"{user_id}\n{user_pw}")

    def perform_login(self, user_id, user_pw, silent=False):
        try:
            self.connection = pymysql.connect(
                host=self.db_host, user=user_id, password=user_pw,
                db='class_dict_db', charset='utf8mb4', cursorclass=pymysql.cursors.DictCursor, autocommit=True
            )
            self.current_user = user_id
            self.show_search_ui()
            return True
        except:
            if not silent: messagebox.showerror("오류", "로그인 정보가 틀립니다.")
            return False

    def show_search_ui(self):
        if hasattr(self, 'login_frame'): self.login_frame.destroy()
        
        self.search_frame = tk.Frame(self.root)
        self.search_frame.pack(pady=10, fill="both", expand=True)
        
        top_frame = tk.Frame(self.search_frame)
        top_frame.pack(fill="x", padx=10)
        tk.Button(top_frame, text="로그아웃", font=("Malgun Gothic", 8), command=self.logout).pack(side="left")
        tk.Label(top_frame, text=f"ID: {self.current_user}", font=self.font_info, fg="gray").pack(side="right")
        
        self.ent_search = tk.Entry(self.search_frame, width=40, font=self.font_main)
        self.ent_search.pack(pady=5)
        self.ent_search.focus_set()
        
        self.list_suggestion = tk.Listbox(self.search_frame, width=40, height=0, font=self.font_main)
        self.list_suggestion.pack(); self.list_suggestion.pack_forget()
        
        self.ent_search.bind('<KeyRelease>', self.on_key_release)
        self.ent_search.bind('<Return>', lambda e: self.search_word())
        self.list_suggestion.bind("<<ListboxSelect>>", self.select_from_list)
        
        btn_frame = tk.Frame(self.search_frame)
        btn_frame.pack(pady=5)
        tk.Button(btn_frame, text="검색", font=self.font_bold, command=self.search_word, width=10).pack(side="left", padx=5)
        tk.Button(btn_frame, text="이전", font=self.font_main, command=self.go_back_history, width=10).pack(side="left", padx=5)
        
        self.txt_result = tk.Text(self.search_frame, height=20, width=60, font=self.font_main, padx=10, pady=10)
        self.txt_result.pack(pady=10, padx=20)
        self.txt_result.tag_config("link", foreground="blue", underline=True)
        self.txt_result.tag_config("info", foreground="gray", font=self.font_info)

        # 수정 버튼 (평소에는 숨김)
        self.btn_edit = tk.Button(self.search_frame, text="📝 이 내용 수정하기", font=self.font_main, 
                                  command=self.open_edit_window, bg="#FFC107")
        self.btn_edit.pack(pady=5)
        self.btn_edit.pack_forget()

        tk.Button(self.search_frame, text="+ 새 용어 등록", font=self.font_main, command=self.open_add_window, bg="#eeeeee").pack()

        # 하단 전광판
        self.ticker_frame = tk.Frame(self.root, bg="#333333", height=35, cursor="hand2")
        self.ticker_frame.pack(side="bottom", fill="x")
        self.lbl_ticker_text = tk.Label(self.ticker_frame, text="최근 등록 단어 로딩 중...", font=self.font_ticker, bg="#333333", fg="white", anchor="w")
        self.lbl_ticker_text.pack(side="left", padx=10, fill="x", expand=True)
        self.lbl_ticker_author = tk.Label(self.ticker_frame, text="", font=self.font_ticker, bg="#333333", fg="#AAAAAA", anchor="e")
        self.lbl_ticker_author.pack(side="right", padx=10)

        for widget in [self.ticker_frame, self.lbl_ticker_text, self.lbl_ticker_author]:
            widget.bind("<Button-1>", lambda e: self.click_ticker())

        self.update_recent_data()
        self.rotate_ticker()

    def search_word(self):
        word = self.ent_search.get().strip()
        self.list_suggestion.pack_forget()
        if not word: return
        try:
            with self.connection.cursor() as cursor:
                sql = "SELECT * FROM terms WHERE word = %s"
                cursor.execute(sql, (word,))
                result = cursor.fetchone()
                self.txt_result.delete(1.0, tk.END)
                
                if result:
                    self.display_with_links(self.txt_result, f"[{word}]\n\n{result.get('definition','')}")
                    self.txt_result.insert(tk.END, f"\n\n{'-'*40}\n✍ {result.get('author','')} | 📅 {result.get('created_at','')}", "info")
                    
                    # ★ 권한 체크: 작성자만 수정 버튼 보이게 함
                    if result.get('author') == self.current_user:
                        self.btn_edit.pack(after=self.txt_result)
                        self.current_editing_data = result # 수정용 데이터 저장
                    else:
                        self.btn_edit.pack_forget()

                    if not self.history or self.history[-1] != word:
                        self.history.append(word)
                        if len(self.history) > 30: self.history.pop(0)
                    self.history_index = len(self.history) - 1
                else:
                    self.txt_result.insert(tk.END, "결과 없음")
                    self.btn_edit.pack_forget()
        except: pass

    def open_edit_window(self):
        """수정 팝업창 열기"""
        self.edit_win = tk.Toplevel(self.root)
        self.edit_win.title("용어 수정 (작성자 전용)")
        self.edit_win.geometry("400x500")
        
        tk.Label(self.edit_win, text=f"단어명: {self.current_editing_data['word']}", font=self.font_bold).pack(pady=10)
        tk.Label(self.edit_win, text="내용 수정:").pack()
        
        self.txt_edit_def = tk.Text(self.edit_win, height=15, width=40, font=self.font_main)
        self.txt_edit_def.pack(pady=10)
        # 기존 내용 채우기
        self.txt_edit_def.insert(tk.END, self.current_editing_data['definition'])
        
        tk.Button(self.edit_win, text="수정 완료", command=self.save_edit_to_db, bg="#4CAF50", fg="white", font=self.font_bold).pack(pady=20)

    def save_edit_to_db(self):
        new_def = self.txt_edit_def.get(1.0, tk.END).strip()
        word = self.current_editing_data['word']
        
        if not new_def: return
        
        try:
            with self.connection.cursor() as cursor:
                sql = "UPDATE terms SET definition = %s, created_at = NOW() WHERE word = %s AND author = %s"
                cursor.execute(sql, (new_def, word, self.current_user))
            messagebox.showinfo("성공", "내용이 수정되었습니다.")
            self.edit_win.destroy()
            self.search_word() # 화면 갱신
            self.update_recent_data() # 전광판 갱신
        except:
            messagebox.showerror("오류", "수정 권한이 없거나 데이터베이스 오류입니다.")

    def on_key_release(self, event):
        if event.keysym == "Down" and self.list_suggestion.winfo_viewable():
            self.list_suggestion.focus_set(); self.list_suggestion.selection_set(0); return
        search_text = self.ent_search.get().strip()
        if not search_text: self.list_suggestion.pack_forget(); return
        try:
            self.connection.ping(reconnect=True)
            with self.connection.cursor() as cursor:
                sql = "SELECT word FROM terms WHERE word LIKE %s LIMIT 5"
                cursor.execute(sql, (f"%{search_text}%",))
                results = cursor.fetchall()
                if results:
                    self.list_suggestion.delete(0, tk.END)
                    for item in results: self.list_suggestion.insert(tk.END, item['word'])
                    self.list_suggestion.config(height=len(results))
                    self.list_suggestion.pack(after=self.ent_search)
                else: self.list_suggestion.pack_forget()
        except: self.list_suggestion.pack_forget()

    def select_from_list(self, event):
        if not self.list_suggestion.curselection(): return
        selected_word = self.list_suggestion.get(self.list_suggestion.curselection())
        self.ent_search.delete(0, tk.END); self.ent_search.insert(0, selected_word)
        self.list_suggestion.pack_forget(); self.search_word()

    def rotate_ticker(self):
        if self.connection and self.recent_terms:
            if self.current_ticker_idx >= len(self.recent_terms): self.current_ticker_idx = 0
            data = self.recent_terms[self.current_ticker_idx]
            word, desc = data['word'], data['definition'].replace('\n', ' ')
            if len(desc) > 30: desc = desc[:30] + "..."
            self.lbl_ticker_text.config(text=f"📢 [최근] {word}: {desc}")
            self.lbl_ticker_author.config(text=f"By: {data['author']}")
            self.last_displayed_word = word 
            self.current_ticker_idx = (self.current_ticker_idx + 1) % len(self.recent_terms)
            self.ticker_timer = self.root.after(5000, self.rotate_ticker)

    def click_ticker(self):
        if hasattr(self, 'last_displayed_word'):
            self.ent_search.delete(0, tk.END); self.ent_search.insert(0, self.last_displayed_word)
            self.search_word()

    def update_recent_data(self):
        try:
            with self.connection.cursor() as cursor:
                sql = "SELECT word, definition, author FROM terms ORDER BY created_at DESC LIMIT 10"
                cursor.execute(sql); self.recent_terms = cursor.fetchall()
        except: pass

    def logout(self):
        if os.path.exists(self.config_file): os.remove(self.config_file)
        if self.ticker_timer: self.root.after_cancel(self.ticker_timer)
        if self.connection: self.connection.close()
        messagebox.showinfo("로그아웃", "로그인 화면으로 돌아갑니다.")
        self.show_login_ui()

    def display_with_links(self, widget, text):
        url_pattern = r'(https?://[^\s]+)'
        parts = re.split(url_pattern, text)
        for part in parts:
            if re.match(url_pattern, part):
                tag = f"link-{part}"
                widget.insert(tk.END, part, ("link", tag))
                widget.tag_bind(tag, "<Button-1>", lambda e, u=part: webbrowser.open(u))
            else: widget.insert(tk.END, part)

    def go_back_history(self):
        if len(self.history) <= 1 or self.history_index <= 0: return
        self.history_index -= 1
        self.ent_search.delete(0, tk.END); self.ent_search.insert(0, self.history[self.history_index])
        self.search_word()

    def open_add_window(self):
        self.add_win = tk.Toplevel(self.root); self.add_win.title("새 용어 등록"); self.add_win.geometry("400x500")
        tk.Label(self.add_win, text="단어명:").pack(); self.ent_new_word = tk.Entry(self.add_win, width=30); self.ent_new_word.pack()
        tk.Label(self.add_win, text="내용:").pack(); self.txt_new_def = tk.Text(self.add_win, height=10, width=40); self.txt_new_def.pack()
        tk.Button(self.add_win, text="저장", command=self.save_to_db, bg="#4CAF50", fg="white").pack(pady=20)

    def save_to_db(self):
        w, d = self.ent_new_word.get().strip(), self.txt_new_def.get(1.0, tk.END).strip()
        if not w or not d: return
        try:
            with self.connection.cursor() as cursor:
                sql = "INSERT INTO terms (word, definition, author, created_at) VALUES (%s, %s, %s, NOW())"
                cursor.execute(sql, (w, d, self.current_user))
            messagebox.showinfo("성공", "등록되었습니다."); self.update_recent_data(); self.add_win.destroy()
        except: messagebox.showerror("오류", "저장 실패")

if __name__ == "__main__":
    root = tk.Tk(); app = DictionaryApp(root); root.mainloop()
